# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [6]:
# imports

# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [13]:
# constants

MODEL_GPT = 'gpt-5-nano'
MODEL_LLAMA = 'llama3.2'
OLLAMA_BASE_URL = 'http://localhost:11434/v1'

In [14]:
# set up environment
system_prompt = """
You are a coding tutor with knowledge of the Python programming language. You can understand the code and explain it in a way that is easy to understand. 
You can also help with other programming languages. you should be able to explain the code in a step by step manner highlighting what each part 
of the code does. you should include some examples to illustrate the concepts.
"""

In [15]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
# Get gpt-4o-mini to answer, with streaming
openai = OpenAI()
stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": question}],
    stream=True,
)
display_handle = display(Markdown(""), display_id=True)
response = ""
for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response), display_id=display_handle.display_id)


What it does

- It builds a set of authors from a collection of books, keeping only non-empty authors and removing duplicates.
- Then it yields each author from that set to the caller (inside a generator) using yield from.

Step-by-step

1) {book.get("author") for book in books if book.get("author")}
   - This is a set comprehension.
   - For each book in books:
     - book.get("author") retrieves the value of the "author" key (or None if missing).
     - The if book.get("author") filters out falsy values (None, empty string, etc.).
   - The resulting set contains unique author values (duplicates collapsed by the set).

2) yield from <set>
   - yield from delegates to the iterable (the set) and yields each element to the caller.
   - Since a set is unordered, the authors will be yielded in an arbitrary order.

Overall effect

- The surrounding generator yields all unique, non-empty authors from the books list. Duplicates are removed; missing/empty authors are ignored. The order is not guaranteed.

Example

- Suppose books = [
    {"title": "A", "author": "Alice"},
    {"title": "B", "author": "Bob"},
    {"title": "C", "author": "Alice"},
    {"title": "D"}  # no author
  ]

- The code yields: "Alice" and "Bob" in some order, e.g. Alice then Bob or Bob then Alice.

Variants and notes

- If you need a deterministic order, you can sort the results, e.g.:
  yield from sorted({book.get("author") for book in books if book.get("author")})

- If you want to preserve insertion order while deduplicating, you’d need a different approach (e.g., track seen authors in a set while iterating and yield as you go).

- This assumes authors are hashable (typical for strings). If author values could be unhashable, the set approach would raise an exception.

In [ ]:
# Get Llama 3.2 to answer
openai = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
stream = openai.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": question}],
    stream=True,
)
display_handle = display(Markdown(""), display_id=True)
response = ""
for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response), display_id=display_handle.display_id)


This code is written in Python and utilizes a concept called "yield from" in combination with a generator expression.

Let's break it down step by step:

- `yield from`: This keyword is used to delegate to a sub-generator. When the `yield from` expression is reached, the execution context shifts to the sub-generator, and the yielded values are sent back to the outer generator.

- `{book.get("author") for book in books if book.get("author")}`: This is a generator expression. It's equivalent to an anonymous function that generates a sequence of values.

  - `for book in books`: This part iterates over a collection of objects (presumably lists or dictionaries) `books`.

  - `if book.get("author")`: This part filters out books that do not have an "author" key.

  - `book.get("author")`: This part extracts the value associated with the key "author" from each book dictionary.

So, when you put it all together, this generator expression does the following:

1. Iterates over each book in the `books` collection.
2. Filters out books that do not have an "author" key.
3. Extracts the author name from each book dictionary without throwing an error if the key does not exist (this is where `.get("author")` comes in, as it returns an empty string if `book.get("author")` does not exist).
4. Yields the extracted author names one by one.

Here's why you would use this code:

If you have a collection of book dictionaries, where each dictionary has an "author" key. You might want to get the names of all the authors from these dictionaries. However, you might not always know how many dictionaries there are or what their structure is like. The `yield from` combination allows you to delegate this operation to a generator expression that can handle this iteration and filtering on the fly, making it very suitable for working with large datasets or data structures where you don't know their layout in advance.

Here's an example of how you might use this code:

```python
books = [
    {"title": "Book A", "author": "Author 1"},
    {"title": "Book B", "author": "Author 2"},
    {"title": "Book C"},
]

author_names = yield from {book.get("author") for book in books if book.get("author")}

for name in author_names:
    print(name)
```

This code will print:

```
Author 1
Author 2
```